# Advanced Feature Engineering

This notebook recreates the base dataset and adds new financial and engagement features to provide more predictive power to the MLP model, breaking the current PR-AUC performance ceiling.

In [ ]:
import logging
import warnings
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

# Constants
TARGET_COL = "Churn"
ID_COL = "CustomerID"
PATH_CUSTOMERS = "../notebooks/data/raw/churn_customers.csv"
PATH_SERVICES = "../notebooks/data/raw/churn_services.csv"
PATH_CONTRACTS = "../notebooks/data/raw/churn_contracts.csv"
PATH_PROCESSED_ADVANCED = "../notebooks/data/processed/churn_processed_advanced.csv"

# Load Data
df_customers = pd.read_csv(PATH_CUSTOMERS)
df_services = pd.read_csv(PATH_SERVICES)
df_contracts = pd.read_csv(PATH_CONTRACTS)

# Standardize IDs and merge
df_customers = df_customers.rename(columns={"customerID": ID_COL})
df_services = df_services.rename(columns={"customerID": ID_COL})
df_contracts = df_contracts.rename(columns={"customerID": ID_COL})

df = (
    df_customers
    .merge(df_services, on=ID_COL, how="inner")
    .merge(df_contracts, on=ID_COL, how="inner")
)

df.rename(columns={"gender": "Gender", "tenure": "Tenure"}, inplace=True)
logger.info(f"Merged Data Shape: {df.shape}")

In [ ]:
# Base Preprocessing
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
median_tc = df["TotalCharges"].median()
df["TotalCharges"] = df["TotalCharges"].fillna(median_tc)

df = df.drop(columns=[ID_COL])

df[TARGET_COL] = df[TARGET_COL].map({"No": 0, "Yes": 1})

binary_yes_no = [
    c for c in df.select_dtypes("object").columns
    if df[c].dropna().isin(["Yes", "No"]).all() and c != TARGET_COL
]

for col in binary_yes_no:
    df[col] = df[col].map({"Yes": 1, "No": 0})


In [ ]:
# Advanced Feature Engineering

# Re-implement previous derived features
df["is_monthly_contract"] = (df["Contract"] == "Month-to-month").astype(int)
df["is_new_customer"] = (df["Tenure"] <= 6).astype(int)

# New Feature 1: charges_per_tenure = TotalCharges / (Tenure + 1)
df["charges_per_tenure"] = df["TotalCharges"] / (df["Tenure"] + 1)

# New Feature 2: is_high_spender = 1 if MonthlyCharges > 75th percentile, else 0
q75_monthly = df["MonthlyCharges"].quantile(0.75)
df["is_high_spender"] = (df["MonthlyCharges"] > q75_monthly).astype(int)

# New Feature 3: total_services_count = sum of flags for 'Yes' in specific services
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df["total_services_count"] = (df[service_cols] == 'Yes').sum(axis=1)

# New Feature 4: has_protection_services = 1 if any of 'Yes' in specific protection services
protection_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']
df["has_protection_services"] = (df[protection_cols] == 'Yes').any(axis=1).astype(int)

logger.info(f"Data Shape after Advanced Feature Engineering: {df.shape}")


In [ ]:
# Sklearn Pipeline and Export

multiclass_cats = [
    c for c in df.select_dtypes("object").columns
    if c not in binary_yes_no and c != TARGET_COL
]

num_features = df.select_dtypes(include=np.number).columns.tolist()
num_features = [c for c in num_features if c != TARGET_COL]

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_features),
        ("cat", cat_pipe, multiclass_cats),
    ],
    remainder="passthrough",
    verbose_feature_names_out=False,
)

X_processed = preprocessor.fit_transform(X)
feature_names_out = preprocessor.get_feature_names_out()
df_processed = pd.DataFrame(X_processed, columns=feature_names_out)
df_processed[TARGET_COL] = y.values

import os
os.makedirs(os.path.dirname(PATH_PROCESSED_ADVANCED), exist_ok=True)
df_processed.to_csv(PATH_PROCESSED_ADVANCED, index=False)
logger.info(f"Advanced Processed Dataset saved to: {PATH_PROCESSED_ADVANCED}")
logger.info(f"Final Shape: {df_processed.shape}")
